# tag_classifier: NLI scoring on a Colab GPU

Runs step 3 (`classify`) for the two zero-shot NLI models, which are too slow on a laptop CPU
(2-4 s per pair). Everything else runs locally.

**No database access is needed here.** The bundle holds only the package code, the cleaned course
texts, the curricula file and the candidate tags; no `.env`.

Before opening this notebook, on your machine:

```
cd pipeline
tag_classifier/.venv/Scripts/python -m tag_classifier.candidates --curriculum <CODE>   # candidates to score
tag_classifier/.venv/Scripts/python -m tag_classifier.bundle                           # -> tag_classifier/data/colab_bundle.zip
```

Then here: Runtime > Change runtime type > T4 GPU, and run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the bundle (`colab_bundle.zip`)

In [ ]:
from google.colab import files
import os, zipfile

os.chdir("/content")
uploaded = files.upload()  # choose tag_classifier/data/colab_bundle.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
print(sorted(os.listdir("/content/tag_classifier/data/candidates")))

## 2. Install (Colab already has torch with CUDA)

In [ ]:
!pip install -q "transformers>=4.48" "sentence-transformers>=3.3"

## 3. Score

Defaults to every curriculum that has a candidate file. Only elective courses are scored
(`electives_only` in `config.toml`). Scores are cached in `tag_classifier/cache/scores.sqlite`;
if the session stops, re-running this cell continues where it left off (the cache is written every
256 pairs).

In [ ]:
import glob, os
CURRICULA = sorted(os.path.basename(p)[:-5] for p in glob.glob("/content/tag_classifier/data/candidates/*.json"))
MODELS = ["modernbert", "bge-m3-zeroshot"]
BATCH = 64
print(CURRICULA)

In [ ]:
for code_ in CURRICULA:
    for model in MODELS:
        !cd /content && python -m tag_classifier.classify --curriculum {code_} --model {model} --batch-size {BATCH}

## 4. Download the scores

In [ ]:
!cd /content && zip -q -r colab_scores.zip tag_classifier/cache/scores.sqlite tag_classifier/data/scores
files.download("/content/colab_scores.zip")

## 5. Back on your machine

Unzip `colab_scores.zip` into `pipeline/tag_classifier/data/colab/<date>/` (git-ignored), then:

```
cd pipeline
tag_classifier/.venv/Scripts/python -m tag_classifier.merge_cache tag_classifier/data/colab/<date>/tag_classifier/cache/scores.sqlite
tag_classifier/.venv/Scripts/python -m tag_classifier.classify --curriculum <CODE>     # all models; NLI pairs come from the cache
tag_classifier/.venv/Scripts/python -m tag_classifier.evaluate --curriculum <CODE>
tag_classifier/.venv/Scripts/python -m tag_classifier.review --curriculum <CODE>
```

`merge_cache` adds the Colab scores without overwriting anything scored locally, and prints how many
pairs of the current candidates are still unscored (non-zero: the bundle was older than the candidates).